# Gene Ribosome Profile from BAM + Zarr

Reconstruct per-gene, per-sample ribosome footprint profiles by:

1. Subsetting `unique_reads.bam` to get reads overlapping a gene
2. Extracting read name → zarr row index (`read_{id}` → row `id`)
3. Inflating per-sample counts from `global_matrix.zarr`
4. Plotting coverage in **spliced transcript coordinates** (introns removed)

In [ ]:
# ── Config ──────────────────────────────────────────────────────────
BAM_PATH  = "/path/to/global/unique_reads.bam"
ZARR_PATH = "/path/to/global/global_matrix.zarr"
GTF_PATH  = "/path/to/annotation.gtf"

OFFSET_FILE = None   # P-site offsets TSV from RiboWaltz (optional)

GENE_NAME = "ACTB"   # lookup by symbol
GENE_ID   = None     # or by Ensembl ID (takes priority)

SAMPLES   = None     # list of SRR ids, or None for all

In [ ]:
import numpy as np
import pysam
import zarr
import matplotlib.pyplot as plt
from collections import defaultdict

## Helper functions

Run this cell once — everything below calls into it.

In [ ]:
# ── GTF parsing ─────────────────────────────────────────────────────

def parse_gtf_attributes(attr_string):
    attrs = {}
    for field in attr_string.strip().split(';'):
        field = field.strip()
        if not field:
            continue
        key, _, value = field.partition(' ')
        attrs[key] = value.strip('"')
    return attrs


def find_gene_in_gtf(gtf_path, gene_name=None, gene_id=None):
    """Return gene dict with chrom/start/end/strand, cds_intervals, exon_intervals."""
    gene_info = None
    cds_intervals, exon_intervals = [], []
    in_gene = False
    target_gene_id = gene_id

    with open(gtf_path) as f:
        for line in f:
            if line.startswith('#'):
                continue
            fields = line.strip().split('\t')
            if len(fields) < 9:
                continue
            feat = fields[2]
            attrs = parse_gtf_attributes(fields[8])

            if feat == 'gene':
                if in_gene:
                    break
                match = (
                    (gene_id and attrs.get('gene_id') == gene_id) or
                    (gene_name and attrs.get('gene_name') == gene_name)
                )
                if match:
                    target_gene_id = attrs.get('gene_id')
                    gene_info = dict(
                        chrom=fields[0],
                        start=int(fields[3]) - 1,
                        end=int(fields[4]),
                        strand=fields[6],
                        gene_id=attrs.get('gene_id', ''),
                        gene_name=attrs.get('gene_name', ''),
                    )
                    in_gene = True
                continue

            if in_gene:
                if attrs.get('gene_id') != target_gene_id:
                    break
                s, e = int(fields[3]) - 1, int(fields[4])
                if feat == 'CDS':
                    cds_intervals.append((s, e))
                elif feat == 'exon':
                    exon_intervals.append((s, e))

    if gene_info is None:
        raise ValueError(f"Gene not found: {gene_name or gene_id}")
    gene_info['cds_intervals'] = sorted(set(cds_intervals))
    gene_info['exon_intervals'] = sorted(set(exon_intervals))
    return gene_info


# ── Transcript coordinate mapping ──────────────────────────────────

def build_tx_coord_map(exon_intervals, cds_intervals, strand):
    """
    Map genomic → spliced transcript coords (5'→3', introns removed).
    Returns (genomic_to_tx dict, tx_length, cds_tx_start, cds_tx_end).
    """
    exon_pos = sorted({p for s, e in sorted(exon_intervals) for p in range(s, e)})
    if strand == '-':
        exon_pos = exon_pos[::-1]

    genomic_to_tx = {gpos: tx for tx, gpos in enumerate(exon_pos)}
    tx_length = len(exon_pos)

    cds_genomic = {p for s, e in cds_intervals for p in range(s, e)}
    cds_tx = sorted(genomic_to_tx[g] for g in cds_genomic if g in genomic_to_tx)
    cds_tx_start = cds_tx[0]  if cds_tx else 0
    cds_tx_end   = cds_tx[-1] + 1 if cds_tx else 0

    return genomic_to_tx, tx_length, cds_tx_start, cds_tx_end


# ── BAM fetching ───────────────────────────────────────────────────

def fetch_gene_reads(bam_path, chrom, start, end, strand=None):
    """Return list of {global_id, positions, is_reverse, aligned_length}."""
    reads = []
    with pysam.AlignmentFile(bam_path, 'rb') as bam:
        for r in bam.fetch(chrom, start, end):
            if r.is_unmapped or r.is_secondary or r.is_supplementary:
                continue
            if strand == '+' and r.is_reverse:
                continue
            if strand == '-' and not r.is_reverse:
                continue
            if not r.query_name.startswith('read_'):
                continue
            positions = r.get_reference_positions()
            if not positions:
                continue
            reads.append(dict(
                global_id=int(r.query_name.split('_')[1]),
                positions=positions,
                is_reverse=r.is_reverse,
                aligned_length=len(positions),
            ))
    return reads


# ── Zarr count inflation ───────────────────────────────────────────

def inflate_counts(gene_reads, counts_arr, sample_idx):
    """Look up per-sample counts for each unique read, batched by zarr chunk."""
    if not gene_reads:
        return {}
    chunk_sz = counts_arr.chunks[0]
    by_chunk = defaultdict(list)
    for gid in sorted({r['global_id'] for r in gene_reads}):
        by_chunk[gid // chunk_sz].append(gid)

    out = {}
    for ci in sorted(by_chunk):
        cs = ci * chunk_sz
        ce = min(cs + chunk_sz, counts_arr.shape[0])
        chunk = counts_arr[cs:ce, :]
        for gid in by_chunk[ci]:
            out[gid] = chunk[gid - cs, sample_idx]
    return out


# ── Profile building ──────────────────────────────────────────────

def load_offsets(path):
    offsets = {}
    with open(path) as f:
        next(f)
        for line in f:
            l, o = line.strip().split('\t')
            offsets[int(l)] = int(o)
    return offsets


def build_profiles(gene_reads, read_counts, n_samples, genomic_to_tx,
                   tx_length, offsets=None):
    """
    Accumulate per-sample coverage in transcript coords.
    If offsets given → A-site mode (single nt per read).
    Otherwise → full footprint coverage.
    """
    profiles = np.zeros((n_samples, tx_length), dtype=np.float64)
    use_asite = offsets is not None

    for rd in gene_reads:
        sc = read_counts.get(rd['global_id'])
        if sc is None:
            continue

        if use_asite:
            ofs = offsets.get(rd['aligned_length'])
            if ofs is None or ofs >= rd['aligned_length']:
                continue
            pos = rd['positions'][ofs] if not rd['is_reverse'] else rd['positions'][-1 - ofs]
            ti = genomic_to_tx.get(pos)
            if ti is not None:
                profiles[:, ti] += sc
        else:
            for pos in rd['positions']:
                ti = genomic_to_tx.get(pos)
                if ti is not None:
                    profiles[:, ti] += sc
    return profiles


# ── Plotting ───────────────────────────────────────────────────────

def plot_profiles(profiles, tx_length, cds_start, cds_end,
                  sample_names, gene_info, max_samples=6, normalise=False):
    """Bar-chart profiles in transcript coords with annotation track."""
    n = min(len(sample_names), max_samples)
    ratios = [3] * n + [0.5]
    fig, axes = plt.subplots(
        n + 1, 1, figsize=(14, 2.2 * n + 1),
        gridspec_kw=dict(height_ratios=ratios, hspace=0.12),
        sharex=True,
    )
    x = np.arange(tx_length)

    for i in range(n):
        ax = axes[i]
        y = profiles[i].copy()
        if normalise and y.sum() > 0:
            y = y / y.sum() * 1e6
        ax.bar(x, y, width=1.0, color='#2c7fb8', edgecolor='none', alpha=0.85)
        ax.axvspan(cds_start, cds_end, alpha=0.06, color='green', zorder=0)
        if cds_start > 0:
            ax.axvline(cds_start, color='green', lw=0.8, ls='--', alpha=0.5)
        if cds_end < tx_length:
            ax.axvline(cds_end, color='red', lw=0.8, ls='--', alpha=0.5)
        ax.set_ylabel('RPM' if normalise else 'Count', fontsize=9)
        ax.set_title(sample_names[i], fontsize=10, loc='left', pad=2)
        ax.spines[['top', 'right']].set_visible(False)

    # Annotation track
    ax_ann = axes[-1]
    ax_ann.plot([0, tx_length], [0.5, 0.5], color='#333', lw=2, solid_capstyle='butt')
    if cds_end > cds_start:
        ax_ann.add_patch(plt.Rectangle(
            (cds_start, 0.2), cds_end - cds_start, 0.6,
            fc='#2c7fb8', ec='#333', lw=1, zorder=3))
        ax_ann.text((cds_start + cds_end) / 2, 0.5, 'CDS',
                    ha='center', va='center', fontsize=9, fontweight='bold',
                    color='white', zorder=4)
    if cds_start > 0:
        ax_ann.text(cds_start / 2, 0.5, "5'UTR",
                    ha='center', va='center', fontsize=8, color='#555')
    if cds_end < tx_length:
        ax_ann.text((cds_end + tx_length) / 2, 0.5, "3'UTR",
                    ha='center', va='center', fontsize=8, color='#555')
    ax_ann.set_xlim(0, tx_length)
    ax_ann.set_ylim(0, 1)
    ax_ann.axis('off')

    axes[-2].set_xlabel("Transcript position (nt, 5'→3')", fontsize=11)
    g = gene_info
    fig.suptitle(
        f"{g['gene_name']} ({g['gene_id']})  {g['chrom']}:{g['start']:,}-{g['end']:,} ({g['strand']})",
        fontsize=13, fontweight='bold')
    plt.tight_layout()
    return fig


print('Helpers loaded ✓')

---
## Load data

In [ ]:
root = zarr.open(ZARR_PATH, mode='r')
counts = root['counts']
sample_names = list(root.attrs['samples'])

print(f"Matrix: {counts.shape[0]:,} reads × {counts.shape[1]:,} samples")

In [ ]:
if SAMPLES:
    sample_idx = [sample_names.index(s) for s in SAMPLES]
    sel_samples = SAMPLES
else:
    sample_idx = list(range(len(sample_names)))
    sel_samples = sample_names

print(f"{len(sel_samples)} samples selected")

## Find gene & build transcript coordinates

In [ ]:
gene = find_gene_in_gtf(GTF_PATH, gene_name=GENE_NAME, gene_id=GENE_ID)

genomic_to_tx, tx_length, cds_tx_start, cds_tx_end = build_tx_coord_map(
    gene['exon_intervals'], gene['cds_intervals'], gene['strand'])

print(f"{gene['gene_name']} ({gene['gene_id']})")
print(f"  {gene['chrom']}:{gene['start']:,}-{gene['end']:,} ({gene['strand']})")
print(f"  Transcript: {tx_length:,} nt  |  CDS: {cds_tx_start}-{cds_tx_end} ({cds_tx_end - cds_tx_start} nt)")

## Fetch reads & inflate counts

In [ ]:
gene_reads = fetch_gene_reads(
    BAM_PATH, gene['chrom'], gene['start'], gene['end'], gene['strand'])
print(f"{len(gene_reads)} unique reads in locus")

read_counts = inflate_counts(gene_reads, counts, sample_idx)

# Per-sample totals
total_per_sample = np.zeros(len(sel_samples), dtype=np.uint64)
for c in read_counts.values():
    total_per_sample += c.astype(np.uint64)

print(f"\nTop samples by count:")
order = np.argsort(total_per_sample)[::-1]
for rank, i in enumerate(order[:8]):
    print(f"  {sel_samples[i]:>15s}  {total_per_sample[i]:>8,}")

## Build profiles

In [ ]:
offsets = load_offsets(OFFSET_FILE) if OFFSET_FILE else None

profiles = build_profiles(
    gene_reads, read_counts, len(sel_samples),
    genomic_to_tx, tx_length, offsets=offsets)

print(f"Mode: {'A-site' if offsets else 'coverage'}")
print(f"Shape: {profiles.shape}")

## Plot

In [ ]:
plot_profiles(profiles, tx_length, cds_tx_start, cds_tx_end,
              sel_samples, gene, max_samples=6);

In [ ]:
plot_profiles(profiles, tx_length, cds_tx_start, cds_tx_end,
              sel_samples, gene, max_samples=6, normalise=True);

## CDS metagene

In [ ]:
N_BINS = 100
cds_len = cds_tx_end - cds_tx_start
cds_profiles = profiles[:, cds_tx_start:cds_tx_end]

meta = np.zeros((len(sel_samples), N_BINS))
for b in range(N_BINS):
    si = int(b * cds_len / N_BINS)
    ei = int((b + 1) * cds_len / N_BINS)
    if ei > si:
        meta[:, b] = cds_profiles[:, si:ei].mean(axis=1)

fig, ax = plt.subplots(figsize=(12, 3.5))
x = np.linspace(0, 100, N_BINS)
for i in range(min(4, len(sel_samples))):
    y = meta[i]
    if y.sum() > 0:
        y = y / y.sum() * 100
    ax.plot(x, y, label=sel_samples[i], alpha=0.8, lw=1.2)
ax.axvline(0, color='green', ls='--', alpha=0.3)
ax.axvline(100, color='red', ls='--', alpha=0.3)
ax.set(xlabel='CDS position (%)', ylabel='Relative coverage (%)',
       title=f"{gene['gene_name']} — CDS metagene")
ax.legend(fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout();

## Reading frame

In [ ]:
fc = np.zeros((len(sel_samples), 3))
for f in range(3):
    fc[:, f] = cds_profiles[:, f::3].sum(axis=1)

n_plots = min(4, len(sel_samples))
fig, axes = plt.subplots(1, n_plots, figsize=(3.5 * n_plots, 3))
if n_plots == 1:
    axes = [axes]
colors = ['#2ecc71', '#f39c12', '#e74c3c']
for i, ax in enumerate(axes):
    tot = fc[i].sum()
    pct = fc[i] / tot * 100 if tot > 0 else fc[i]
    ax.bar([0, 1, 2], pct, color=colors, ec='black', lw=0.5)
    ax.set(xticks=[0, 1, 2], xticklabels=['F0', 'F1', 'F2'],
           ylabel='% CDS reads', ylim=(0, 100), title=sel_samples[i])
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle(f"{gene['gene_name']} — Reading Frame", fontweight='bold')
plt.tight_layout();

## Summary

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'sample': sel_samples,
    'total_reads': total_per_sample,
    'frame0_pct': np.round(fc[:, 0] / fc.sum(axis=1).clip(1) * 100, 1),
    'frame1_pct': np.round(fc[:, 1] / fc.sum(axis=1).clip(1) * 100, 1),
    'frame2_pct': np.round(fc[:, 2] / fc.sum(axis=1).clip(1) * 100, 1),
}).sort_values('total_reads', ascending=False)

print(f"{gene['gene_name']} — {len(gene_reads)} unique reads")
df.head(20)

## Gene × Sample count matrix (all genes)

Iterate over every gene in the GTF, fetch overlapping reads from the BAM,
inflate counts from zarr → produces a `(genes × samples)` raw count matrix.

**Note:** This scans the full BAM once per gene. For a large GTF this takes
a while — progress is printed every 500 genes.

In [ ]:
import time

# ── 1. Parse all genes from GTF ────────────────────────────────────
def parse_all_genes(gtf_path):
    """Return list of (gene_id, gene_name, chrom, start, end, strand)."""
    genes = []
    with open(gtf_path) as f:
        for line in f:
            if line.startswith('#'):
                continue
            fields = line.strip().split('\t')
            if len(fields) < 9 or fields[2] != 'gene':
                continue
            attrs = parse_gtf_attributes(fields[8])
            genes.append((
                attrs.get('gene_id', ''),
                attrs.get('gene_name', ''),
                fields[0],                   # chrom
                int(fields[3]) - 1,          # start (0-based)
                int(fields[4]),              # end
                fields[6],                   # strand
            ))
    return genes

all_genes = parse_all_genes(GTF_PATH)
print(f"{len(all_genes)} genes in GTF")

# ── 2. Count reads per gene per sample ─────────────────────────────
bam = pysam.AlignmentFile(BAM_PATH, 'rb')
bam_chroms = set(bam.references)

n_samples = len(sel_samples)
gene_ids, gene_names = [], []
count_matrix = []

t0 = time.time()
for gi, (gid, gname, chrom, start, end, strand) in enumerate(all_genes):
    if chrom not in bam_chroms:
        continue

    # Fetch reads overlapping this gene
    read_global_ids = set()
    for r in bam.fetch(chrom, start, end):
        if r.is_unmapped or r.is_secondary or r.is_supplementary:
            continue
        if strand == '+' and r.is_reverse:
            continue
        if strand == '-' and not r.is_reverse:
            continue
        name = r.query_name
        if name.startswith('read_'):
            read_global_ids.add(int(name.split('_')[1]))

    if not read_global_ids:
        continue

    # Inflate from zarr — batch by chunk
    chunk_sz = counts.chunks[0]
    by_chunk = defaultdict(list)
    for rid in sorted(read_global_ids):
        by_chunk[rid // chunk_sz].append(rid)

    gene_total = np.zeros(n_samples, dtype=np.uint64)
    for ci in sorted(by_chunk):
        cs = ci * chunk_sz
        ce = min(cs + chunk_sz, counts.shape[0])
        chunk = counts[cs:ce, :]
        for rid in by_chunk[ci]:
            gene_total += chunk[rid - cs, sample_idx].astype(np.uint64)

    gene_ids.append(gid)
    gene_names.append(gname)
    count_matrix.append(gene_total)

    if (gi + 1) % 500 == 0:
        elapsed = time.time() - t0
        print(f"  {gi+1}/{len(all_genes)} genes  ({len(gene_ids)} with reads)  [{elapsed:.0f}s]")

bam.close()
elapsed = time.time() - t0
print(f"\nDone: {len(gene_ids)} genes with ≥1 read  [{elapsed:.0f}s]")

# ── 3. Assemble into DataFrame ─────────────────────────────────────
count_arr = np.array(count_matrix, dtype=np.uint32)  # (genes × samples)

gene_count_df = pd.DataFrame(
    count_arr,
    index=pd.MultiIndex.from_arrays([gene_ids, gene_names], names=['gene_id', 'gene_name']),
    columns=sel_samples,
)

print(f"Matrix: {gene_count_df.shape[0]} genes × {gene_count_df.shape[1]} samples")
gene_count_df.head(10)

In [ ]:
# Save to TSV
out_path = str(Path(ZARR_PATH).parent / "gene_counts_matrix.tsv")
gene_count_df.to_csv(out_path, sep='\t')
print(f"Saved to {out_path}")